In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path('../data')

device_catalog = pd.read_csv(BASE_DIR / 'device_catalog.csv')
device_sessions = pd.read_csv(BASE_DIR / 'device_sessions.csv')
dts_holdout = pd.read_csv(BASE_DIR / 'dts_holdout.csv')
dts_train = pd.read_csv(BASE_DIR / 'dts_train.csv')
kyc_records = pd.read_csv(BASE_DIR / 'kyc_records.csv')
sim_events = pd.read_csv(BASE_DIR / 'sim_events.csv')

In [27]:
datasets = {
    "device_catalog": device_catalog,
    "device_sessions": device_sessions,
    "dts_holdout": dts_holdout,
    "dts_train": dts_train,
    "kyc_records": kyc_records,
    "sim_events": sim_events,
}

summary = []

for name, df in datasets.items():
    summary.append({
        "table": name,
        "rows": len(df),
        "cols": len(df.columns)
    })

pd.DataFrame(summary)

,table,rows,cols
0,device_catalog,18,6
1,device_sessions,1276727,9
2,dts_holdout,20000,57
3,dts_train,51047,60
4,kyc_records,71047,5
5,sim_events,101755,4


In [24]:
def quick_look(df):
    print("Info:")
    df.info()
    print()
    print("Sample:")
    print(df.head())

In [31]:
for name, dataset in datasets.items():
    print(name)
    quick_look(dataset)
    print()

device_catalog
Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   TAC                18 non-null     int64 
 1   DeviceBrand        18 non-null     object
 2   DeviceModel        18 non-null     object
 3   DeviceOS           18 non-null     object
 4   DeviceReleaseYear  18 non-null     int64 
 5   DeviceTier         18 non-null     int64 
dtypes: int64(3), object(3)
memory usage: 992.0+ bytes

Sample:
        TAC DeviceBrand DeviceModel DeviceOS  DeviceReleaseYear  DeviceTier
0  35201105     Samsung  Galaxy S21  Android               2021           1
1  35420911     Samsung  Galaxy A52  Android               2021           2
2  35884303     Samsung   Galaxy J7  Android               2017           3
3  35330814       Apple   iPhone 13      iOS               2021           1
4  35699207       Apple   iPhone 11      iOS             

In [47]:
for name, df in datasets.items():
    print()
    print(name)
    print(df.columns.tolist())


device_catalog
['TAC', 'DeviceBrand', 'DeviceModel', 'DeviceOS', 'DeviceReleaseYear', 'DeviceTier']

device_sessions
['CustomerID', 'SessionDate', 'SessionHour', 'IMEI', 'IP', 'IPType', 'CountryCode', 'IsHomeCell', 'RootedFlag']

dts_holdout
['CustomerID', 'MonthlyRevenue', 'MonthlyMinutes', 'TotalRecurringCharge', 'DirectorAssistedCalls', 'OverageMinutes', 'RoamingCalls', 'PercChangeMinutes', 'PercChangeRevenues', 'DroppedCalls', 'BlockedCalls', 'UnansweredCalls', 'CustomerCareCalls', 'ThreewayCalls', 'ReceivedCalls', 'OutboundCalls', 'InboundCalls', 'PeakCallsInOut', 'OffPeakCallsInOut', 'DroppedBlockedCalls', 'CallForwardingCalls', 'CallWaitingCalls', 'MonthsInService', 'UniqueSubs', 'ActiveSubs', 'ServiceArea', 'Handsets', 'HandsetModels', 'CurrentEquipmentDays', 'AgeHH1', 'AgeHH2', 'ChildrenInHH', 'HandsetRefurbished', 'HandsetWebCapable', 'TruckOwner', 'RVOwner', 'Homeownership', 'BuysViaMailOrder', 'RespondsToMailOffers', 'OptOutMailings', 'NonUSTravel', 'OwnsComputer', 'HasCre

In [36]:
len(
    set(dts_train["CustomerID"])
    &
    set(dts_holdout["CustomerID"])
)

0

In [37]:
dts_train["FraudFlag"].value_counts(normalize=True)

FraudFlag
0    0.96801
1    0.03199
Name: proportion, dtype: float64

In [44]:
print(device_catalog.columns.to_list())
print(device_sessions.columns.to_list())

['TAC', 'DeviceBrand', 'DeviceModel', 'DeviceOS', 'DeviceReleaseYear', 'DeviceTier']
['CustomerID', 'SessionDate', 'SessionHour', 'IMEI', 'IP', 'IPType', 'CountryCode', 'IsHomeCell', 'RootedFlag']


In [53]:
device_sessions['TAC'] = (
    device_sessions['IMEI']
    .astype(str)
    .str[:8]
)

device_catalog["TAC"] = (
    device_catalog["TAC"]
    .astype(str)
)

In [54]:
device_sessions = device_sessions.merge(
    device_catalog,
    on="TAC",
    how="left"
)

In [58]:
# 1 IMEI có bao nhiêu account 
device_sessions.groupby("IMEI")["CustomerID"].nunique().describe()

count    74536.000000
mean         1.031247
std          0.257398
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          8.000000
Name: CustomerID, dtype: float64

In [59]:
# 1 customer có bao nhiêu IMEI
device_sessions.groupby("CustomerID")["IMEI"].nunique().describe()

count    71047.000000
mean         1.081889
std          0.274198
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
Name: IMEI, dtype: float64

In [60]:
# 1 customer có bao nhiêu SIM
sim_events.groupby("CustomerID")["ICCID"].nunique().describe()

count    71047.000000
mean         1.330894
std          0.566124
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          6.000000
Name: ICCID, dtype: float64

In [62]:
for name, df in datasets.items():
    missing = (
        df.isna()
        .mean()
        .sort_values(ascending=False)
    )
    
    print(name)
    print(missing.head(20))
    print()

device_catalog
TAC                  0.0
DeviceBrand          0.0
DeviceModel          0.0
DeviceOS             0.0
DeviceReleaseYear    0.0
DeviceTier           0.0
dtype: float64

device_sessions
CustomerID     0.0
SessionDate    0.0
SessionHour    0.0
IMEI           0.0
IP             0.0
IPType         0.0
CountryCode    0.0
IsHomeCell     0.0
RootedFlag     0.0
TAC            0.0
dtype: float64

dts_holdout
AgeHH2                   0.01675
AgeHH1                   0.01675
PercChangeMinutes        0.00675
PercChangeRevenues       0.00675
MonthlyMinutes           0.00300
TotalRecurringCharge     0.00300
DirectorAssistedCalls    0.00300
OverageMinutes           0.00300
RoamingCalls             0.00300
MonthlyRevenue           0.00300
ServiceArea              0.00020
CustomerID               0.00000
OptOutMailings           0.00000
OwnsComputer             0.00000
NonUSTravel              0.00000
BuysViaMailOrder         0.00000
RespondsToMailOffers     0.00000
RetentionCalls          